In [ ]:
# Pipeline smoke test: when True, every expensive stage uses tiny data and few steps so you can verify the full notebook end-to-end quickly.
# Set to False for real benchmarks and training.
smalltest = True

### Quick test mode

The cell above sets `smalltest`. When it is `True`, the **Experiment config** cell applies minimal data and short runs across **all** stages (SCROLLS + WikiText, GPT-2/Pythia baselines, sliding-window, HFOLD, exports). Use `False` for full benchmarks.


## Standalone setup (run first)

Same pattern as `baselines.ipynb`: clone the repo if needed, move into it on the `pranav` branch, then install Python dependencies. After this section, the rest of the notebook runs from the repository root (`mls/`), so paths like `data/` and `adewan_branch/MLSFINAL 2/` resolve correctly.

If you already opened this notebook from a local clone, you can still run these cells; they are idempotent (skip clone if `mls` already exists).

**Push to this repo (`pranav` branch):** from the `mls` repo root, add the remote once if needed (`git remote add mls https://github.com/pranavr11/mls.git`), then `git push mls pranav`.

In [ ]:
%%bash
set -e
if [ ! -d mls ]; then
  git clone --branch pranav https://github.com/pranavr11/mls.git
fi

In [ ]:
%cd mls
!git checkout pranav

In [ ]:
!pip install -q -r requirements.txt
# Trainer / HF stack extras used by this notebook (not all in requirements.txt on older pins).
!pip install -q accelerate sentencepiece

# HFOLD End-to-End Benchmark Notebook

Run the cells **top to bottom**. The first section clones `https://github.com/pranavr11/mls` (branch `pranav`), changes into `mls/`, and installs `requirements.txt` plus notebook extras—same pattern as `baselines.ipynb`, so you can use this notebook standalone in a fresh folder.

This notebook builds a reproducible pipeline for:

1. **Normal attention baselines** on SCROLLS (excluding `narrative_qa`) and WikiText.
2. **Sliding-window attention baselines** (fine-tuned with local attention active).
3. **HFOLD model creation, fine-tuning, and evaluation** on the same corpora.

The workflow is intentionally explicit and step-by-step:

- Set global config + deterministic seeds.
- Load all SCROLLS tasks except `narrative_qa` + full WikiText.
- Build causal-LM training/eval corpora.
- Fine-tune/evaluate full-attention GPT-2 + Pythia-160M.
- Fine-tune/evaluate sliding-window GPT-2 + Pythia-160M.
- Train/evaluate HFOLD model.
- Produce a unified comparison table.

> **Compute note:** for a **full pipeline smoke test** (tiny data + few steps everywhere), set `smalltest = True` in the first code cell. For lighter debugging without the full `smalltest` profile, set `EXPERIMENT.quick_debug = True` in the config cell. For production benchmarks, use `smalltest = False` and `quick_debug = False`.


In [ ]:
# Dependencies: installed in the setup cells above (`requirements.txt` + accelerate/sentencepiece).
# Uncomment only if you need to upgrade versions mid-session:
# !pip install -U datasets evaluate transformers accelerate sentencepiece

In [ ]:
import gc
import math
import os
import random
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, concatenate_datasets, get_dataset_config_names, load_dataset
from evaluate import load as load_metric
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
)


@dataclass
class ExperimentConfig:
    # Repro + I/O
    seed: int = 42
    cache_dir: str = "./data"
    output_root: str = "./outputs/hfold_notebook"
    quick_debug: bool = False

    # Data
    scrolls_dataset_name: str = "tau/scrolls"
    scrolls_exclude_tasks: Tuple[str, ...] = ("narrative_qa",)
    wikitext_dataset_name: str = "wikitext"
    wikitext_config_name: str = "wikitext-103-raw-v1"

    # Models for full/sliding baselines
    baseline_models: Tuple[str, ...] = ("gpt2", "EleutherAI/pythia-160m")

    # Tokenization / sequence shaping
    block_size: int = 512
    # Keep prompts <= block_size so HFOLD and baselines use comparable context.
    generation_prompt_max_length: int = 512
    generation_max_new_tokens: int = 128

    # Full/sliding fine-tuning
    epochs: float = 1.0
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    logging_steps: int = 20
    save_steps: int = 500
    eval_steps: int = 500

    # Sliding window
    sliding_window_size: int = 256

    # Evaluation controls
    latency_trials: int = 3
    max_scrolls_eval_samples_per_task: Optional[int] = None
    max_wikitext_eval_samples: Optional[int] = None

    # HFOLD training
    hfold_d_model: int = 768
    hfold_n_heads: int = 12
    hfold_n_layers: int = 6
    hfold_d_ff: int = 3072
    hfold_window_size: int = 128
    hfold_heap_size: int = 64
    hfold_q_topk: int = 16
    hfold_e_pop: int = 8
    hfold_dropout: float = 0.1
    hfold_epochs: int = 1
    hfold_learning_rate: float = 3e-4
    hfold_weight_decay: float = 0.01
    hfold_grad_clip: float = 1.0
    hfold_train_batch_size: int = 2
    hfold_eval_batch_size: int = 2

    # Optional caps (used by `smalltest` / manual runs)
    scrolls_tasks_filter: Optional[Tuple[str, ...]] = None
    wikitext_train_max_docs: Optional[int] = None
    wikitext_val_max_docs: Optional[int] = None
    scrolls_train_max_rows: Optional[int] = None
    scrolls_val_max_rows: Optional[int] = None
    baseline_max_steps: Optional[int] = None
    hfold_max_train_batches: Optional[int] = None
    hfold_max_eval_batches: Optional[int] = None


EXPERIMENT = ExperimentConfig()

_smalltest = bool(globals().get("smalltest", False))
if _smalltest:
    EXPERIMENT.output_root = "./outputs/hfold_notebook_smalltest"
    EXPERIMENT.scrolls_tasks_filter = ("gov_report",)
    EXPERIMENT.wikitext_train_max_docs = 200
    EXPERIMENT.wikitext_val_max_docs = 40
    EXPERIMENT.scrolls_train_max_rows = 48
    EXPERIMENT.scrolls_val_max_rows = 16
    EXPERIMENT.max_scrolls_eval_samples_per_task = 1
    EXPERIMENT.max_wikitext_eval_samples = 12
    EXPERIMENT.block_size = 128
    EXPERIMENT.generation_prompt_max_length = 128
    EXPERIMENT.generation_max_new_tokens = 24
    EXPERIMENT.latency_trials = 1
    EXPERIMENT.per_device_train_batch_size = 1
    EXPERIMENT.per_device_eval_batch_size = 1
    EXPERIMENT.gradient_accumulation_steps = 1
    EXPERIMENT.logging_steps = 1
    EXPERIMENT.eval_steps = 99999
    EXPERIMENT.save_steps = 99999
    EXPERIMENT.baseline_max_steps = 8
    EXPERIMENT.hfold_d_model = 256
    EXPERIMENT.hfold_n_heads = 4
    EXPERIMENT.hfold_n_layers = 2
    EXPERIMENT.hfold_d_ff = 1024
    EXPERIMENT.hfold_window_size = 32
    EXPERIMENT.hfold_heap_size = 8
    EXPERIMENT.hfold_q_topk = 2
    EXPERIMENT.hfold_e_pop = 2
    EXPERIMENT.hfold_epochs = 1
    EXPERIMENT.hfold_train_batch_size = 1
    EXPERIMENT.hfold_eval_batch_size = 1
    EXPERIMENT.hfold_max_train_batches = 12
    EXPERIMENT.hfold_max_eval_batches = 4
elif EXPERIMENT.quick_debug:
    EXPERIMENT.max_scrolls_eval_samples_per_task = 16
    EXPERIMENT.max_wikitext_eval_samples = 512
    EXPERIMENT.epochs = 0.1
    EXPERIMENT.hfold_epochs = 1

Path(EXPERIMENT.output_root).mkdir(parents=True, exist_ok=True)
if _smalltest:
    print("*** smalltest=True: minimal data and short runs for end-to-end smoke test ***")
print("Experiment config:")
print(pd.Series(asdict(EXPERIMENT)))

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


set_seed(EXPERIMENT.seed)
DEVICE = pick_device()
print(f"Using device: {DEVICE}")

## Step 1 - Load SCROLLS (minus NarrativeQA) and WikiText

This stage downloads and caches:

- all SCROLLS task configs except `narrative_qa`
- full WikiText-103 raw dataset

It also prepares helper structures for later training/evaluation.

In [ ]:
def get_scrolls_task_names() -> List[str]:
    all_tasks = get_dataset_config_names(EXPERIMENT.scrolls_dataset_name)
    excluded = set(EXPERIMENT.scrolls_exclude_tasks)
    kept = sorted([name for name in all_tasks if name not in excluded])
    return kept


def load_scrolls_tasks(task_names: List[str]) -> Dict[str, DatasetDict]:
    scrolls_tasks = {}
    for task_name in tqdm(task_names, desc="Loading SCROLLS tasks"):
        scrolls_tasks[task_name] = load_dataset(
            EXPERIMENT.scrolls_dataset_name,
            task_name,
            cache_dir=EXPERIMENT.cache_dir,
            trust_remote_code=True,
        )
    return scrolls_tasks


def load_wikitext() -> DatasetDict:
    return load_dataset(
        EXPERIMENT.wikitext_dataset_name,
        EXPERIMENT.wikitext_config_name,
        cache_dir=EXPERIMENT.cache_dir,
    )


scrolls_task_names = get_scrolls_task_names()
if EXPERIMENT.scrolls_tasks_filter is not None:
    allowed = set(EXPERIMENT.scrolls_tasks_filter)
    scrolls_task_names = [n for n in scrolls_task_names if n in allowed]
    print("SCROLLS tasks after filter:", EXPERIMENT.scrolls_tasks_filter)
print(f"SCROLLS tasks kept ({len(scrolls_task_names)}):")
print(scrolls_task_names)

scrolls_raw = load_scrolls_tasks(scrolls_task_names)
wikitext_raw = load_wikitext()

print("\nLoaded SCROLLS split sizes:")
for name, ds in scrolls_raw.items():
    sizes = {split: len(split_ds) for split, split_ds in ds.items()}
    print(f"- {name}: {sizes}")

print("\nLoaded WikiText split sizes:")
print({split: len(split_ds) for split, split_ds in wikitext_raw.items()})

## Step 2 - Build training/evaluation corpora

We create:

- a **joint causal-LM training corpus** (`WikiText + SCROLLS`) for fine-tuning,
- a **SCROLLS generation eval set** per task (for ROUGE),
- a **WikiText eval text set** (for perplexity).

In [ ]:
def normalize_scrolls_reference(output_field: Any) -> str:
    if isinstance(output_field, list):
        return "\n".join([str(x) for x in output_field])
    return str(output_field)


def scrolls_example_to_lm_text(task_name: str, item: Dict[str, Any]) -> str:
    source = str(item["input"])
    target = normalize_scrolls_reference(item["output"])
    return (
        f"### Task: {task_name}\n"
        f"### Input:\n{source}\n"
        f"### Response:\n{target}"
    )


def build_scrolls_lm_corpus(scrolls_tasks: Dict[str, DatasetDict], split: str) -> Dataset:
    all_rows = []
    for task_name, ds in scrolls_tasks.items():
        if split not in ds:
            continue
        for row in ds[split]:
            all_rows.append({"text": scrolls_example_to_lm_text(task_name, row)})
    return Dataset.from_list(all_rows)


def build_scrolls_generation_eval_rows(
    scrolls_tasks: Dict[str, DatasetDict],
    split: str = "validation",
    max_per_task: Optional[int] = None,
) -> Dict[str, List[Dict[str, str]]]:
    task_rows: Dict[str, List[Dict[str, str]]] = {}
    for task_name, ds in scrolls_tasks.items():
        if split not in ds:
            continue
        rows = []
        for idx, row in enumerate(ds[split]):
            if max_per_task is not None and idx >= max_per_task:
                break
            rows.append(
                {
                    "input": str(row["input"]),
                    "reference": normalize_scrolls_reference(row["output"]),
                }
            )
        task_rows[task_name] = rows
    return task_rows


def build_wikitext_text_dataset(dataset_split: Dataset) -> Dataset:
    texts = [t for t in dataset_split["text"] if isinstance(t, str) and t.strip()]
    return Dataset.from_dict({"text": texts})


scrolls_train_text = build_scrolls_lm_corpus(scrolls_raw, split="train")
scrolls_val_text = build_scrolls_lm_corpus(scrolls_raw, split="validation")

wikitext_train_text = build_wikitext_text_dataset(wikitext_raw["train"])
wikitext_val_text = build_wikitext_text_dataset(wikitext_raw["validation"])

if EXPERIMENT.wikitext_train_max_docs is not None:
    n = min(len(wikitext_train_text), EXPERIMENT.wikitext_train_max_docs)
    wikitext_train_text = wikitext_train_text.select(range(n))
if EXPERIMENT.wikitext_val_max_docs is not None:
    n = min(len(wikitext_val_text), EXPERIMENT.wikitext_val_max_docs)
    wikitext_val_text = wikitext_val_text.select(range(n))
if EXPERIMENT.scrolls_train_max_rows is not None:
    n = min(len(scrolls_train_text), EXPERIMENT.scrolls_train_max_rows)
    scrolls_train_text = scrolls_train_text.select(range(n))
if EXPERIMENT.scrolls_val_max_rows is not None:
    n = min(len(scrolls_val_text), EXPERIMENT.scrolls_val_max_rows)
    scrolls_val_text = scrolls_val_text.select(range(n))

joint_train_text = concatenate_datasets([wikitext_train_text, scrolls_train_text])
joint_val_text = concatenate_datasets([wikitext_val_text, scrolls_val_text])

scrolls_eval_rows = build_scrolls_generation_eval_rows(
    scrolls_raw,
    split="validation",
    max_per_task=EXPERIMENT.max_scrolls_eval_samples_per_task,
)

wikitext_eval_text = wikitext_val_text
if EXPERIMENT.max_wikitext_eval_samples is not None:
    capped = min(len(wikitext_eval_text), EXPERIMENT.max_wikitext_eval_samples)
    wikitext_eval_text = wikitext_eval_text.select(range(capped))

print("Corpus sizes:")
print(f"- WikiText train texts: {len(wikitext_train_text):,}")
print(f"- SCROLLS train texts: {len(scrolls_train_text):,}")
print(f"- Joint train texts: {len(joint_train_text):,}")
print(f"- Joint validation texts: {len(joint_val_text):,}")
print(f"- WikiText eval texts: {len(wikitext_eval_text):,}")
print("- SCROLLS eval rows per task:")
print({k: len(v) for k, v in scrolls_eval_rows.items()})

In [ ]:
def ensure_pad_token(tokenizer) -> None:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token


def build_clm_chunks(dataset: Dataset, tokenizer, block_size: int) -> Dataset:
    def tokenize_batch(examples: Dict[str, List[str]]) -> Dict[str, List[List[int]]]:
        return tokenizer(examples["text"], add_special_tokens=False)

    tokenized = dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing text",
    )

    def group_texts(examples: Dict[str, List[List[int]]]) -> Dict[str, List[List[int]]]:
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated["input_ids"])
        total_length = (total_length // block_size) * block_size

        if total_length == 0:
            return {"input_ids": [], "attention_mask": []}

        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated.items()
        }
        return result

    chunks = tokenized.map(group_texts, batched=True, desc="Grouping into fixed-length CLM chunks")
    return chunks


def build_training_components(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ensure_pad_token(tokenizer)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    train_chunks = build_clm_chunks(joint_train_text, tokenizer, EXPERIMENT.block_size)
    val_chunks = build_clm_chunks(joint_val_text, tokenizer, EXPERIMENT.block_size)
    wiki_eval_chunks = build_clm_chunks(wikitext_eval_text, tokenizer, EXPERIMENT.block_size)

    return tokenizer, data_collator, train_chunks, val_chunks, wiki_eval_chunks


def generate_prediction(model, tokenizer, prompt: str, max_new_tokens: int, is_hfold: bool = False) -> str:
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=EXPERIMENT.generation_prompt_max_length,
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        if is_hfold:
            output_ids = model.generate(
                encoded["input_ids"],
                max_new_tokens=max_new_tokens,
                temperature=1.0,
                top_k=50,
                top_p=0.95,
            )
        else:
            output_ids = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
            )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def evaluate_scrolls_rouge(model, tokenizer, eval_rows_by_task: Dict[str, List[Dict[str, str]]], is_hfold: bool = False) -> pd.DataFrame:
    rouge_metric = load_metric("rouge")
    rows = []

    for task_name, rows_for_task in tqdm(eval_rows_by_task.items(), desc="ROUGE on SCROLLS tasks"):
        predictions = []
        references = []

        for row in rows_for_task:
            pred = generate_prediction(
                model,
                tokenizer,
                prompt=row["input"],
                max_new_tokens=EXPERIMENT.generation_max_new_tokens,
                is_hfold=is_hfold,
            )
            predictions.append(pred)
            references.append(row["reference"])

        if len(predictions) == 0:
            rows.append(
                {
                    "task": task_name,
                    "rouge1": float("nan"),
                    "rouge2": float("nan"),
                    "rougeL": float("nan"),
                    "rougeLsum": float("nan"),
                    "num_examples": 0,
                }
            )
            continue

        scores = rouge_metric.compute(predictions=predictions, references=references)
        rows.append(
            {
                "task": task_name,
                "rouge1": scores["rouge1"],
                "rouge2": scores["rouge2"],
                "rougeL": scores["rougeL"],
                "rougeLsum": scores["rougeLsum"],
                "num_examples": len(rows_for_task),
            }
        )

    return pd.DataFrame(rows).sort_values("task").reset_index(drop=True)


def benchmark_generation_latency(model, tokenizer, prompts: List[str], is_hfold: bool = False) -> Dict[str, float]:
    measured_latencies = []
    generated_token_counts = []

    for prompt in prompts:
        encoded = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=EXPERIMENT.generation_prompt_max_length,
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        # Warm-up
        with torch.no_grad():
            if is_hfold:
                warm_out = model.generate(encoded["input_ids"], max_new_tokens=EXPERIMENT.generation_max_new_tokens)
            else:
                warm_out = model.generate(
                    **encoded,
                    max_new_tokens=EXPERIMENT.generation_max_new_tokens,
                    pad_token_id=tokenizer.eos_token_id,
                )

        for _ in range(EXPERIMENT.latency_trials):
            start = time.time()
            with torch.no_grad():
                if is_hfold:
                    out = model.generate(encoded["input_ids"], max_new_tokens=EXPERIMENT.generation_max_new_tokens)
                else:
                    out = model.generate(
                        **encoded,
                        max_new_tokens=EXPERIMENT.generation_max_new_tokens,
                        pad_token_id=tokenizer.eos_token_id,
                    )
            end = time.time()
            measured_latencies.append(end - start)
            generated_token_counts.append(out.shape[1] - encoded["input_ids"].shape[1])

    avg_latency = float(np.mean(measured_latencies)) if measured_latencies else float("nan")
    avg_generated_tokens = float(np.mean(generated_token_counts)) if generated_token_counts else float("nan")
    tokens_per_sec = avg_generated_tokens / avg_latency if avg_latency > 0 else float("nan")

    return {
        "avg_latency_sec": avg_latency,
        "avg_generated_tokens": avg_generated_tokens,
        "tokens_per_sec": tokens_per_sec,
        "num_prompt_cases": len(prompts),
    }


def estimate_attention_flops_per_token(seq_len: int, d_model: int, n_heads: int, mode: str, window_size: int = 0, e_pop: int = 0) -> float:
    d_k = d_model // n_heads
    if mode == "full":
        return 2.0 * seq_len * seq_len * d_k * n_heads
    if mode == "sliding":
        return 2.0 * seq_len * window_size * d_k * n_heads
    if mode == "hfold":
        return 2.0 * seq_len * (window_size + e_pop) * d_k * n_heads
    raise ValueError(f"Unknown mode: {mode}")

## Step 3 - Attention module implementations

This cell defines:

- normal/full attention (default pretrained model behavior)
- sliding-window attention patching for GPT-2 and Pythia (GPT-NeoX family)

The sliding-window patch changes the active attention pattern during both training and inference.

In [ ]:
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

try:
    from transformers.models.gpt_neox.modeling_gpt_neox import GPTNeoXAttention
    HAS_GPT_NEOX_ATTENTION = True
except Exception as err:
    print(f"GPTNeoXAttention import unavailable: {err}")
    GPTNeoXAttention = None
    HAS_GPT_NEOX_ATTENTION = False


def make_sliding_window_causal_mask(seq_len: int, window_size: int, device: torch.device) -> torch.Tensor:
    positions = torch.arange(seq_len, device=device)
    i = positions[:, None]
    j = positions[None, :]
    return (j <= i) & (j >= i - window_size + 1)


def make_query_key_sliding_mask(query_len: int, key_len: int, window_size: int, device: torch.device) -> torch.Tensor:
    if query_len == key_len:
        return make_sliding_window_causal_mask(query_len, window_size, device)
    full = make_sliding_window_causal_mask(key_len, window_size, device)
    return full[-query_len:, :]


class SlidingWindowGPT2Attention(GPT2Attention):
    def __init__(self, config, is_cross_attention: bool = False, layer_idx: Optional[int] = None, window_size: int = 64):
        super().__init__(config, is_cross_attention=is_cross_attention, layer_idx=layer_idx)
        self.window_size = window_size

    def _attn(self, query, key, value, attention_mask=None, head_mask=None, **kwargs):
        scores = torch.matmul(query, key.transpose(-1, -2))

        if getattr(self, "scale_attn_weights", True):
            scores = scores / (value.size(-1) ** 0.5)

        if getattr(self, "scale_attn_by_inverse_layer_idx", False):
            layer_idx = getattr(self, "layer_idx", 0)
            scores = scores / float(layer_idx + 1)

        q_len = query.size(-2)
        k_len = key.size(-2)
        sw_mask_2d = make_query_key_sliding_mask(q_len, k_len, self.window_size, scores.device)
        sw_mask = sw_mask_2d[None, None, :, :]

        scores = scores.masked_fill(~sw_mask, torch.finfo(scores.dtype).min)

        if attention_mask is not None:
            scores = scores + attention_mask

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = attn_weights.type(value.dtype)
        attn_weights = self.attn_dropout(attn_weights)

        if head_mask is not None:
            attn_weights = attn_weights * head_mask

        attn_output = torch.matmul(attn_weights, value)
        return attn_output, attn_weights


def replace_gpt2_attention_with_sliding_window(model, window_size: int):
    for layer_idx, block in enumerate(model.transformer.h):
        old_attn = block.attn
        new_attn = SlidingWindowGPT2Attention(
            model.config,
            is_cross_attention=getattr(old_attn, "is_cross_attention", False),
            layer_idx=getattr(old_attn, "layer_idx", layer_idx),
            window_size=window_size,
        )
        new_attn.load_state_dict(old_attn.state_dict(), strict=False)

        ref_param = next(old_attn.parameters(), None)
        if ref_param is not None:
            new_attn.to(device=ref_param.device, dtype=ref_param.dtype)

        block.attn = new_attn
    return model


if HAS_GPT_NEOX_ATTENTION:
    class SlidingWindowGPTNeoXAttention(GPTNeoXAttention):
        def __init__(self, config, layer_idx: Optional[int] = None, window_size: int = 64):
            super().__init__(config, layer_idx=layer_idx)
            self.window_size = window_size

        def _attn(self, query, key, value, attention_mask=None, head_mask=None, **kwargs):
            scores = torch.matmul(query, key.transpose(-1, -2))
            norm_factor = getattr(self, "norm_factor", value.size(-1) ** 0.5)
            scores = scores / norm_factor

            q_len = query.size(-2)
            k_len = key.size(-2)
            sw_mask_2d = make_query_key_sliding_mask(q_len, k_len, self.window_size, scores.device)
            sw_mask = sw_mask_2d[None, None, :, :]
            scores = scores.masked_fill(~sw_mask, torch.finfo(scores.dtype).min)

            if attention_mask is not None:
                scores = scores + attention_mask

            attn_weights = torch.softmax(scores, dim=-1)
            attn_weights = attn_weights.type(value.dtype)

            dropout = getattr(self, "attention_dropout", None)
            if dropout is None:
                dropout = getattr(self, "attn_dropout")
            attn_weights = dropout(attn_weights)

            if head_mask is not None:
                attn_weights = attn_weights * head_mask

            attn_output = torch.matmul(attn_weights, value)
            return attn_output, attn_weights


    def replace_pythia_attention_with_sliding_window(model, window_size: int):
        if not hasattr(model, "gpt_neox"):
            raise ValueError("Expected GPT-NeoX architecture for Pythia sliding-window patch.")

        for layer_idx, block in enumerate(model.gpt_neox.layers):
            old_attn = block.attention
            new_attn = SlidingWindowGPTNeoXAttention(
                model.config,
                layer_idx=getattr(old_attn, "layer_idx", layer_idx),
                window_size=window_size,
            )
            new_attn.load_state_dict(old_attn.state_dict(), strict=False)

            ref_param = next(old_attn.parameters(), None)
            if ref_param is not None:
                new_attn.to(device=ref_param.device, dtype=ref_param.dtype)

            block.attention = new_attn

        return model
else:
    def replace_pythia_attention_with_sliding_window(model, window_size: int):
        raise RuntimeError("GPTNeoXAttention is not available in this transformers build.")


def build_lm_model(model_name: str, attention_mode: str):
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if attention_mode == "full":
        pass
    elif attention_mode == "sliding":
        if model_name == "gpt2":
            model = replace_gpt2_attention_with_sliding_window(model, EXPERIMENT.sliding_window_size)
        elif "pythia" in model_name.lower():
            model = replace_pythia_attention_with_sliding_window(model, EXPERIMENT.sliding_window_size)
        else:
            raise ValueError(f"Sliding-window patch not implemented for model: {model_name}")
    else:
        raise ValueError(f"Unsupported attention mode: {attention_mode}")

    if model.config.pad_token_id is None and model.config.eos_token_id is not None:
        model.config.pad_token_id = model.config.eos_token_id

    model.to(DEVICE)
    return model

## Step 4 - Fine-tune + evaluate normal and sliding-window baselines

This block runs both attention modes for each baseline model:

- `full` (normal pretrained attention)
- `sliding` (patched local causal attention)

For each run it records:

- WikiText perplexity
- SCROLLS ROUGE (per task)
- generation latency/tokens-per-second
- attention FLOP estimate (for comparison)

In [ ]:
def build_training_args(run_dir: str) -> TrainingArguments:
    fp16 = torch.cuda.is_available()
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    kwargs = dict(
        output_dir=run_dir,
        overwrite_output_dir=True,
        num_train_epochs=EXPERIMENT.epochs,
        learning_rate=EXPERIMENT.learning_rate,
        weight_decay=EXPERIMENT.weight_decay,
        warmup_ratio=EXPERIMENT.warmup_ratio,
        per_device_train_batch_size=EXPERIMENT.per_device_train_batch_size,
        per_device_eval_batch_size=EXPERIMENT.per_device_eval_batch_size,
        gradient_accumulation_steps=EXPERIMENT.gradient_accumulation_steps,
        logging_steps=EXPERIMENT.logging_steps,
        eval_steps=EXPERIMENT.eval_steps,
        save_steps=EXPERIMENT.save_steps,
        save_total_limit=2,
        report_to="none",
        fp16=fp16 and not bf16,
        bf16=bf16,
        dataloader_num_workers=0 if EXPERIMENT.baseline_max_steps is not None else 2,
        remove_unused_columns=False,
    )

    ta_fields = TrainingArguments.__dataclass_fields__
    if EXPERIMENT.baseline_max_steps is not None:
        kwargs["max_steps"] = EXPERIMENT.baseline_max_steps
        if "evaluation_strategy" in ta_fields:
            kwargs["evaluation_strategy"] = "no"
        else:
            kwargs["eval_strategy"] = "no"
        kwargs["save_steps"] = 10**9
    else:
        if "evaluation_strategy" in ta_fields:
            kwargs["evaluation_strategy"] = "steps"
        else:
            kwargs["eval_strategy"] = "steps"
        if "save_strategy" in ta_fields:
            kwargs["save_strategy"] = "steps"

    return TrainingArguments(**kwargs)


def compute_wikitext_perplexity_with_trainer(trainer: Trainer, wiki_eval_chunks: Dataset) -> float:
    eval_metrics = trainer.evaluate(eval_dataset=wiki_eval_chunks)
    eval_loss = eval_metrics.get("eval_loss", None)
    if eval_loss is None:
        return float("nan")
    try:
        return float(math.exp(eval_loss))
    except OverflowError:
        return float("inf")


def run_baseline_experiment(model_name: str, attention_mode: str) -> Dict[str, Any]:
    run_name = f"{model_name.replace('/', '_')}__{attention_mode}"
    run_dir = str(Path(EXPERIMENT.output_root) / run_name)
    print(f"\n=== Running {run_name} ===")

    tokenizer, data_collator, train_chunks, val_chunks, wiki_eval_chunks = build_training_components(model_name)
    model = build_lm_model(model_name, attention_mode)

    training_args = build_training_args(run_dir)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_chunks,
        eval_dataset=val_chunks,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    train_result = trainer.train()
    trainer.save_model(run_dir)
    tokenizer.save_pretrained(run_dir)

    wiki_perplexity = compute_wikitext_perplexity_with_trainer(trainer, wiki_eval_chunks)
    scrolls_rouge_df = evaluate_scrolls_rouge(model, tokenizer, scrolls_eval_rows, is_hfold=False)

    latency_prompts = []
    # one prompt from WikiText
    if len(wikitext_eval_text) > 0:
        latency_prompts.append(wikitext_eval_text[0]["text"])
    # one prompt per SCROLLS task
    for task_name, rows in scrolls_eval_rows.items():
        if rows:
            latency_prompts.append(rows[0]["input"])

    latency_metrics = benchmark_generation_latency(model, tokenizer, latency_prompts, is_hfold=False)

    seq_len_for_flops = EXPERIMENT.block_size
    if "pythia" in model_name.lower():
        d_model = 768
        n_heads = 12
    else:
        d_model = 768
        n_heads = 12

    flop_mode = "full" if attention_mode == "full" else "sliding"
    flops = estimate_attention_flops_per_token(
        seq_len=seq_len_for_flops,
        d_model=d_model,
        n_heads=n_heads,
        mode=flop_mode,
        window_size=EXPERIMENT.sliding_window_size,
    )

    summary = {
        "model": model_name,
        "attention_mode": attention_mode,
        "train_runtime_sec": train_result.metrics.get("train_runtime", float("nan")),
        "train_steps_per_sec": train_result.metrics.get("train_steps_per_second", float("nan")),
        "wikitext_perplexity": wiki_perplexity,
        "scrolls_rouge1_mean": float(scrolls_rouge_df["rouge1"].mean()),
        "scrolls_rouge2_mean": float(scrolls_rouge_df["rouge2"].mean()),
        "scrolls_rougeL_mean": float(scrolls_rouge_df["rougeL"].mean()),
        "latency_avg_sec": latency_metrics["avg_latency_sec"],
        "latency_tokens_per_sec": latency_metrics["tokens_per_sec"],
        "attention_flops_estimate_per_sequence": flops,
        "output_dir": run_dir,
    }

    scrolls_rouge_path = Path(run_dir) / "scrolls_rouge_by_task.csv"
    scrolls_rouge_df.to_csv(scrolls_rouge_path, index=False)

    del trainer
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


baseline_summaries = []
for baseline_model in EXPERIMENT.baseline_models:
    for mode in ("full", "sliding"):
        if mode == "sliding" and ("pythia" in baseline_model.lower()) and (not HAS_GPT_NEOX_ATTENTION):
            print(f"Skipping {baseline_model} sliding run because GPTNeoXAttention is unavailable.")
            continue
        baseline_summaries.append(run_baseline_experiment(baseline_model, mode))

baseline_results_df = pd.DataFrame(baseline_summaries)
baseline_results_df

## Step 5 - Create, fine-tune, and test HFOLD

This section uses the project HFOLD implementation from `adewan_branch/MLSFINAL 2/hfold`.

We train HFOLD on the same joint corpus and evaluate with:

- WikiText perplexity
- SCROLLS ROUGE
- generation latency
- HFOLD attention FLOP estimate

In [ ]:
HFOLD_PACKAGE_ROOT = Path("adewan_branch") / "MLSFINAL 2"
if not HFOLD_PACKAGE_ROOT.exists():
    raise FileNotFoundError(
        f"Expected HFOLD package path not found: {HFOLD_PACKAGE_ROOT}. "
        "Make sure the adewan branch snapshot exists in the workspace."
    )

if str(HFOLD_PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(HFOLD_PACKAGE_ROOT))

from hfold.core.config import HFoldConfig as ProjectHFoldConfig
from hfold.models.hfold_transformer import HFoldTransformer

print("Imported HFOLD implementation from:", HFOLD_PACKAGE_ROOT)


# HFOLD will use GPT-2 tokenizer vocabulary so comparisons are easier.
hfold_tokenizer = AutoTokenizer.from_pretrained("gpt2")
ensure_pad_token(hfold_tokenizer)

hfold_train_chunks = build_clm_chunks(joint_train_text, hfold_tokenizer, EXPERIMENT.block_size)
hfold_val_chunks = build_clm_chunks(joint_val_text, hfold_tokenizer, EXPERIMENT.block_size)
hfold_wiki_eval_chunks = build_clm_chunks(wikitext_eval_text, hfold_tokenizer, EXPERIMENT.block_size)

print(f"HFOLD train chunks: {len(hfold_train_chunks):,}")
print(f"HFOLD val chunks: {len(hfold_val_chunks):,}")
print(f"HFOLD WikiText eval chunks: {len(hfold_wiki_eval_chunks):,}")

In [ ]:
def clm_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    input_ids = torch.tensor([row["input_ids"] for row in batch], dtype=torch.long)
    attention_mask = None
    if "attention_mask" in batch[0]:
        attention_mask = torch.tensor([row["attention_mask"] for row in batch], dtype=torch.long)

    result = {"input_ids": input_ids}
    if attention_mask is not None:
        result["attention_mask"] = attention_mask
    return result


def build_hfold_model(tokenizer) -> HFoldTransformer:
    hfold_max_seq_len = max(EXPERIMENT.block_size + EXPERIMENT.generation_max_new_tokens, 2048)
    cfg = ProjectHFoldConfig(
        vocab_size=len(tokenizer),
        d_model=EXPERIMENT.hfold_d_model,
        n_heads=EXPERIMENT.hfold_n_heads,
        n_layers=EXPERIMENT.hfold_n_layers,
        d_ff=EXPERIMENT.hfold_d_ff,
        max_seq_len=hfold_max_seq_len,
        window_size=EXPERIMENT.hfold_window_size,
        heap_size=EXPERIMENT.hfold_heap_size,
        q_topk=EXPERIMENT.hfold_q_topk,
        e_pop=EXPERIMENT.hfold_e_pop,
        dropout=EXPERIMENT.hfold_dropout,
    )
    model = HFoldTransformer(cfg)
    model.to(DEVICE)
    return model


def compute_hfold_loss(logits: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
    # Standard next-token causal LM loss.
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    vocab = shift_logits.size(-1)
    return F.cross_entropy(shift_logits.view(-1, vocab), shift_labels.view(-1))


def train_hfold_model(model: HFoldTransformer, train_dataset: Dataset, val_dataset: Dataset) -> Dict[str, float]:
    train_loader = DataLoader(
        train_dataset,
        batch_size=EXPERIMENT.hfold_train_batch_size,
        shuffle=True,
        collate_fn=clm_collate,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=EXPERIMENT.hfold_eval_batch_size,
        shuffle=False,
        collate_fn=clm_collate,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=EXPERIMENT.hfold_learning_rate,
        weight_decay=EXPERIMENT.hfold_weight_decay,
    )

    global_step = 0
    model.train()
    start_time = time.time()

    for epoch in range(EXPERIMENT.hfold_epochs):
        epoch_losses = []
        pbar = tqdm(train_loader, desc=f"HFOLD train epoch {epoch + 1}/{EXPERIMENT.hfold_epochs}")

        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(DEVICE)
            outputs = model(input_ids, return_logits=True, return_heaps=False)
            loss = compute_hfold_loss(outputs["logits"], input_ids)
            loss = loss / EXPERIMENT.gradient_accumulation_steps
            loss.backward()

            if (step + 1) % EXPERIMENT.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), EXPERIMENT.hfold_grad_clip)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

            epoch_losses.append(loss.item() * EXPERIMENT.gradient_accumulation_steps)
            pbar.set_postfix(loss=float(np.mean(epoch_losses[-20:])))
            if EXPERIMENT.hfold_max_train_batches is not None and (step + 1) >= EXPERIMENT.hfold_max_train_batches:
                break

    train_runtime = time.time() - start_time

    # validation loss/perplexity
    model.eval()
    val_losses = []
    with torch.no_grad():
        for vi, batch in enumerate(tqdm(val_loader, desc="HFOLD validation")):
            input_ids = batch["input_ids"].to(DEVICE)
            outputs = model(input_ids, return_logits=True, return_heaps=False)
            loss = compute_hfold_loss(outputs["logits"], input_ids)
            val_losses.append(loss.item())
            if EXPERIMENT.hfold_max_eval_batches is not None and (vi + 1) >= EXPERIMENT.hfold_max_eval_batches:
                break

    val_loss = float(np.mean(val_losses)) if val_losses else float("nan")
    val_perplexity = float(math.exp(val_loss)) if np.isfinite(val_loss) else float("nan")

    return {
        "train_runtime_sec": train_runtime,
        "train_global_steps": global_step,
        "val_loss": val_loss,
        "val_perplexity": val_perplexity,
    }


def compute_hfold_wikitext_perplexity(model: HFoldTransformer, wiki_chunks: Dataset) -> float:
    eval_loader = DataLoader(
        wiki_chunks,
        batch_size=EXPERIMENT.hfold_eval_batch_size,
        shuffle=False,
        collate_fn=clm_collate,
    )

    model.eval()
    losses = []
    with torch.no_grad():
        for wi, batch in enumerate(tqdm(eval_loader, desc="HFOLD WikiText perplexity")):
            input_ids = batch["input_ids"].to(DEVICE)
            outputs = model(input_ids, return_logits=True, return_heaps=False)
            losses.append(compute_hfold_loss(outputs["logits"], input_ids).item())
            if EXPERIMENT.hfold_max_eval_batches is not None and (wi + 1) >= EXPERIMENT.hfold_max_eval_batches:
                break

    if not losses:
        return float("nan")
    mean_loss = float(np.mean(losses))
    return float(math.exp(mean_loss))

In [ ]:
hfold_model = build_hfold_model(hfold_tokenizer)

hfold_train_metrics = train_hfold_model(hfold_model, hfold_train_chunks, hfold_val_chunks)
hfold_wikitext_perplexity = compute_hfold_wikitext_perplexity(hfold_model, hfold_wiki_eval_chunks)
hfold_scrolls_rouge_df = evaluate_scrolls_rouge(hfold_model, hfold_tokenizer, scrolls_eval_rows, is_hfold=True)

hfold_latency_prompts = []
if len(wikitext_eval_text) > 0:
    hfold_latency_prompts.append(wikitext_eval_text[0]["text"])
for task_name, rows in scrolls_eval_rows.items():
    if rows:
        hfold_latency_prompts.append(rows[0]["input"])

hfold_latency_metrics = benchmark_generation_latency(
    hfold_model,
    hfold_tokenizer,
    hfold_latency_prompts,
    is_hfold=True,
)

hfold_run_dir = Path(EXPERIMENT.output_root) / "hfold_model"
hfold_run_dir.mkdir(parents=True, exist_ok=True)
torch.save(hfold_model.state_dict(), hfold_run_dir / "hfold_state_dict.pt")
hfold_tokenizer.save_pretrained(hfold_run_dir)
hfold_scrolls_rouge_df.to_csv(hfold_run_dir / "scrolls_rouge_by_task.csv", index=False)

hfold_flops = estimate_attention_flops_per_token(
    seq_len=EXPERIMENT.block_size,
    d_model=EXPERIMENT.hfold_d_model,
    n_heads=EXPERIMENT.hfold_n_heads,
    mode="hfold",
    window_size=EXPERIMENT.hfold_window_size,
    e_pop=EXPERIMENT.hfold_e_pop,
)

hfold_summary = {
    "model": "HFOLD",
    "attention_mode": "hfold",
    "train_runtime_sec": hfold_train_metrics["train_runtime_sec"],
    "train_steps_per_sec": (
        hfold_train_metrics["train_global_steps"] / hfold_train_metrics["train_runtime_sec"]
        if hfold_train_metrics["train_runtime_sec"] > 0
        else float("nan")
    ),
    "wikitext_perplexity": hfold_wikitext_perplexity,
    "scrolls_rouge1_mean": float(hfold_scrolls_rouge_df["rouge1"].mean()),
    "scrolls_rouge2_mean": float(hfold_scrolls_rouge_df["rouge2"].mean()),
    "scrolls_rougeL_mean": float(hfold_scrolls_rouge_df["rougeL"].mean()),
    "latency_avg_sec": hfold_latency_metrics["avg_latency_sec"],
    "latency_tokens_per_sec": hfold_latency_metrics["tokens_per_sec"],
    "attention_flops_estimate_per_sequence": hfold_flops,
    "output_dir": str(hfold_run_dir),
}

pd.DataFrame([hfold_summary])

## Step 6 - Final comparison table + exports

This combines all baseline runs and HFOLD into one report-ready dataframe.

In [ ]:
all_results_df = pd.concat(
    [
        baseline_results_df,
        pd.DataFrame([hfold_summary]),
    ],
    ignore_index=True,
)

all_results_df = all_results_df[
    [
        "model",
        "attention_mode",
        "wikitext_perplexity",
        "scrolls_rouge1_mean",
        "scrolls_rouge2_mean",
        "scrolls_rougeL_mean",
        "latency_avg_sec",
        "latency_tokens_per_sec",
        "attention_flops_estimate_per_sequence",
        "train_runtime_sec",
        "train_steps_per_sec",
        "output_dir",
    ]
]

all_results_path = Path(EXPERIMENT.output_root) / "all_results_summary.csv"
all_results_df.to_csv(all_results_path, index=False)

print(f"Saved final summary to: {all_results_path}")
all_results_df.sort_values(["model", "attention_mode"]).reset_index(drop=True)